# ️ Glu-Stock: 04_BACKTEST_LAB
**Phase**: Historical Performance Analysis | v18.23 (Profit Engine)

This notebook simulates signals and emulates trade exits to estimate Profit/Loss and Holding Periods for the 2025 trading year.

In [ ]:
#!pip install -q yfinance pandas scikit-learn joblib tensorflow ta lightgbm


In [ ]:
import os, json, joblib, numpy as np, pandas as pd, yfinance as yf, warnings, ta
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

def find_model_file(filename):
    search_paths = ['/kaggle/input', '/kaggle/working', '.', 'data/models']
    for root_dir in search_paths:
        if not os.path.exists(root_dir): continue
        for root, dirs, files in os.walk(root_dir):
            if filename in files: return os.path.join(root, filename)
    return None

def frac_diff(series, d=0.4, window=100):
    w = [1.0]
    for k in range(1, window):
        w.append(-w[-1] * (d - k + 1) / k)
    w = np.array(w[::-1])
    result = np.full(len(series), np.nan)
    for t in range(window - 1, len(series)):
        result[t] = np.dot(w, series[t - window + 1:t + 1])
    return result

class MLPredictor:
    def __init__(self, path):
        brain = joblib.load(path)
        self.model = brain.get('model')
        self.features = brain.get('features', [])

    def build_features(self, df):
        close = df['Close'].squeeze()
        high = df['High'].squeeze()
        low = df['Low'].squeeze()
        volume = df['Volume'].squeeze()
        feat = pd.DataFrame(index=df.index)
        feat['Returns'] = close.pct_change()
        feat['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi()
        macd = ta.trend.MACD(close=close)
        feat['MACD'] = macd.macd_diff()
        boll = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
        feat['BB_High'] = boll.bollinger_hband_indicator()
        feat['BB_Low'] = boll.bollinger_lband_indicator()
        feat['ATR'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
        feat['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14).adx()
        obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
        feat['OBV_norm'] = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-7)
        feat['day_of_week'] = df.index.dayofweek
        feat['week_of_month'] = (df.index.day - 1) // 7
        feat['frac_diff_close'] = frac_diff(close.values.flatten(), d=0.4, window=100)
        vol_ma = volume.rolling(20).mean()
        feat['vol_ratio'] = volume / (vol_ma + 1e-7)
        return feat.dropna()

    def predict(self, df):
        try:
            feat = self.build_features(df)
            if len(feat) == 0: return 0.0
            row = feat[self.features].tail(1)
            proba = self.model.predict_proba(row)[0]
            return float(proba[1])
        except: return 0.0

class CNNPredictor:
    def __init__(self, path):
        self.interpreter = tflite.Interpreter(model_path=path)
        self.interpreter.allocate_tensors()

    def predict(self, df):
        try:
            close = df['Close'].squeeze().values[-30:]
            high = df['High'].squeeze().values[-30:]
            low = df['Low'].squeeze().values[-30:]
            volume = df['Volume'].squeeze().values[-30:]
            opn = df['Open'].squeeze().values[-30:]
            raw = np.column_stack([opn, high, low, close, volume])
            seq_min, seq_max = raw.min(axis=0), raw.max(axis=0)
            norm_seq = (raw - seq_min) / (seq_max - seq_min + 1e-7)
            input_details = self.interpreter.get_input_details()
            input_data = np.expand_dims(norm_seq.astype(np.float32), axis=0)
            self.interpreter.set_tensor(input_details[0]['index'], input_data)
            self.interpreter.invoke()
            output = self.interpreter.get_tensor(self.interpreter.get_output_details()[0]['index'])[0]
            return float(output[1]) if len(output) > 1 else float(output[0])
        except: return 0.5


In [ ]:
def simulate_trade_exit(future_df, entry_price, tp=0.03, sl=0.02, horizon=10):
    """Emulates a Triple Barrier Exit Strategy"""
    if len(future_df) == 0: return 0.0, 0
    
    tp_price = entry_price * (1 + tp)
    sl_price = entry_price * (1 - sl)
    
    for i in range(min(len(future_df), horizon)):
        high = future_df['High'].iloc[i]
        low = future_df['Low'].iloc[i]
        close = future_df['Close'].iloc[i]
        
        if high >= tp_price:
            return tp * 100, i + 1 # Profit Hit
        if low <= sl_price:
            return -sl * 100, i + 1 # Loss Hit
            
    # Time Exit
    final_price = future_df['Close'].iloc[min(len(future_df)-1, horizon-1)]
    final_return = (final_price / entry_price - 1) * 100
    return final_return, min(len(future_df), horizon)

def run_performance_backtest():
    print("Starting 2025 Performance Analysis (v18.23)...\n")
    
    lgbm_path = find_model_file('glu_brain_v1.joblib')
    cnn_path = find_model_file('cnn_daily_t2.tflite')
    if not lgbm_path or not cnn_path: return
    
    lgbm = MLPredictor(lgbm_path)
    cnn = CNNPredictor(cnn_path)
    
    cohort = ['BBCA.JK', 'TLKM.JK', 'ASII.JK', 'ADRO.JK', 'BMRI.JK', 'BBRI.JK', 'ICBP.JK', 'PTBA.JK', 'ANTM.JK', 'UNTR.JK']
    data = yf.download(cohort + ['^JKSE'], start='2024-06-01', end='2025-12-31', progress=False, auto_adjust=True)
    
    valid_dates = [d for d in data.index if d.year == 2025]
    signals_found = []
    
    print(f"Scanning {len(valid_dates)} trading days for signals...")
    for date in valid_dates[::1]: # Daily scan
        for ticker in cohort:
            try:
                # 1. Historical Context
                hist = data.loc[:date, (slice(None), ticker)]
                hist.columns = hist.columns.droplevel(1)
                hist = hist.dropna()
                if len(hist) < 150: continue
                
                # Filter: SMA 50 Guard
                if hist['Close'].iloc[-1] < hist['Close'].tail(50).mean(): continue
                
                # 2. ML Inference
                l_prob = lgbm.predict(hist)
                c_prob = cnn.predict(hist)
                score = (l_prob * 0.4) + (c_prob * 0.6)
                
                if score >= 0.5:
                    entry_price = float(hist['Close'].iloc[-1])
                    # 3. Future Performance
                    future = data.loc[date + timedelta(days=1):, (slice(None), ticker)]
                    future.columns = future.columns.droplevel(1)
                    
                    profit, hold_days = simulate_trade_exit(future, entry_price)
                    
                    signals_found.append({
                        'Date': date.strftime('%Y-%m-%d'),
                        'Ticker': ticker,
                        'Score': f"{score:.2%}",
                        'Entry': f"{entry_price:,.0f}",
                        'Profit%': round(profit, 2),
                        'Days': hold_days
                    })
            except: continue
            
    results = pd.DataFrame(signals_found)
    print(f"\n[SUCCESS] Simulation Complete.")
    print(f"Total Signals Found in 2025: {len(results)}")
    
    if not results.empty:
        win_rate = (results['Profit%'] > 0).mean() * 100
        avg_profit = results['Profit%'].mean()
        print(f"--- RECAP PERFORMANCE ---")
        print(f"Win Rate: {win_rate:.2f}%")
        print(f"Avg Profit: {avg_profit:.2f}%")
        print(f"Avg Hold: {results['Days'].mean():.1f} Days")
        print("\n--- Detailed Results (First 50) ---")
        print(results.head(50).to_string(index=False))
    else:
        print("\n[FAIL] No signals found in 2025.")

run_performance_backtest()
